# License

In [ ]:
# ------------------------------------------------------------------------------
#  QuantumThermal sample, written in Python
#
#  Multi-Scale Modeling Laboratory (SMaLL)
#  Website: https://small.polito.it/
#  GitHub repository: https://github.com/SMaLL-PoliTo/QuantumThermal
#
#  Copyright (C) 2026 Pietro Asinari, Matteo Maria Piredda,
#  Giulio Barletta
#  E-mail contact: pietro.asinari@polito.it
#
#  This code is licensed under the MIT License.
#  You may obtain a copy of the License at
#
#      https://opensource.org/licenses/MIT
#
#  Permission is hereby granted, free of charge, to any person obtaining a copy
#  of this software and associated documentation files (the "Software"), to deal
#  in the Software without restriction, including without limitation the rights
#  to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
#  copies of the Software, and to permit persons to whom the Software is
#  furnished to do so, subject to the following conditions:
#
#  The above copyright notice and this permission notice shall be included in
#  all copies or substantial portions of the Software.
#
#  THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
#  IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
#  FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
#  AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
#  LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
#  OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
#  SOFTWARE.
# ------------------------------------------------------------------------------

# Real Data Loading in Qiskit for a 3-Qubit Register

This notebook implements a **divide-and-conquer state-preparation routine** for loading a **real-valued amplitude vector** into a **3-qubit quantum circuit** using **Qiskit**.

The notebook is meant as a practical companion to the discussion on **real data loading/encoding** in the appendix of our HHL tutorial paper. Here, the goal is to show, step by step, how a classical real vector with $(2^3 = 8)$ entries can be mapped to the amplitudes of a 3-qubit quantum state.

The implementation follows the approach of:

*Araujo I., Park D., Petruccione F., da Silva A. J., "A divide-and-conquer algorithm for quantum state preparation", Scientific Reports 11, 6329 (2021).*

## What this notebook does

1. defines the recursive routine that computes the rotation angles;
2. builds the 3-qubit state-preparation circuit;
3. inspects the ideal statevector produced by the circuit;
4. runs a shot-based simulation and compares the sampled output distribution with the target one.

## Important convention used here

The input list contains **amplitudes**, not probabilities.  
Therefore, if the target vector is

$x = [x_0, x_1, \, \ldots, x_7]$

then the prepared quantum state is

$|\psi\rangle = \sum_{i=0}^{7} x_i |i\rangle$

with the usual normalization condition $\sum_i x_i^2 = 1$.

# Preparation

This notebook was tested with **Python 3.11** and **Qiskit 1.4.2**.

Before running the notebook, create and activate a dedicated conda environment:

```conda create -n thermal-science -y python=3.11 ipykernel```

```conda activate thermal-science```

The next cell installs the packages used in this example.


In [ ]:
%pip install numpy scipy matplotlib cvxpy pylatexenc
%pip install qiskit==1.4.2 qiskit-aer==0.17.2 qiskit-algorithms==0.3.1

## Step 1, define the routines used for state preparation

The divide-and-conquer strategy works in two stages:

- `gen_angles(x)` computes the sequence of rotation angles required to encode the target real vector `x`;
- `gen_circuit(angles)` uses those angles to assemble the corresponding **3-qubit Qiskit circuit**.

The main idea is recursive:

- pairs of amplitudes are grouped together;
- each pair is associated with a rotation angle;
- the same procedure is then repeated on the reduced vector obtained from the pairwise norms.


In [ ]:
import numpy as np
from qiskit.circuit import QuantumCircuit
from qiskit.circuit.library import RYGate
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.visualization import plot_histogram

In [ ]:
# Reference:
# Araujo I., Park D., Petruccione F., da Silva A. J.
# "A divide-and-conquer algorithm for quantum state preparation"
# Scientific Reports 11, 6329 (2021).
def gen_angles(x):
    """Recursively compute the rotation angles needed for amplitude encoding.

    Parameters
    ----------
    x : array-like
        Real-valued normalized amplitude vector of length 2^n.

    Returns
    -------
    numpy.ndarray
        Array of rotation angles used to build the state-preparation circuit.
    """
    N = int(len(x))
    Nd2 = int(N / 2)
    new_x = np.zeros(Nd2)

    if N > 1:
        # Pair neighboring amplitudes and compute the norm of each pair.
        # These norms define the reduced vector used by the recursive step.
        for k in range(0, Nd2):
            new_x[k] = np.sqrt(x[2 * k] ** 2 + x[2 * k + 1] ** 2)

        # Recursive call on the reduced vector
        inner_angles = gen_angles(new_x)
        angles = np.zeros(Nd2)

        # For each pair (x[2k], x[2k+1]), determine the Y-rotation angle
        # that reproduces the relative weight between the |0> and |1> branch.
        for k in range(0, Nd2):
            if new_x[k] != 0:
                if x[2 * k] > 0:
                    angles[k] = 2 * np.asin(x[2 * k + 1] / new_x[k])
                else:
                    angles[k] = 2 * np.pi - 2 * np.asin(x[2 * k + 1] / new_x[k])
            else:
                angles[k] = 0

        # For n > 1 qubits, prepend the angles coming from the coarser levels
        # of the recursion tree.
        if N > 2:
            angles = np.concatenate((inner_angles, angles))

        return angles

print("Function defined: gen_angles(x)")
print("This function takes a real normalized vector and returns the rotation angles.")

def gen_circuit(angles):
    """Build the 3-qubit state-preparation circuit from the rotation angles."""
    N = int(len(angles)) + 1
    n = np.log2(N)

    circuit = QuantumCircuit(n)

    # General template retained for consistency with the original implementation.
    # The inactive branches below are intentionally left untouched.
    if 1 > 2:
        circuit.cry(angles[1], 0, 1, ctrl_state="0")

    if 1 > 2:
        circuit.cry(angles[2], 0, 1)

    if 1 > 2:
        c2ry = RYGate(angles[3]).control(2, ctrl_state="00")
        circuit.append(c2ry, [0, 1, 2])

        c2ry = RYGate(angles[4]).control(2, ctrl_state="10")
        circuit.append(c2ry, [0, 1, 2])

        c2ry = RYGate(angles[5]).control(2, ctrl_state="01")
        circuit.append(c2ry, [0, 1, 2])

        c2ry = RYGate(angles[6]).control(2)
        circuit.append(c2ry, [0, 1, 2])

    # Active 3-qubit implementation used in this notebook
    #
    # Qubit roles in this circuit:
    # - qubit 2: first split of the recursion tree
    # - qubit 1: second split, conditioned on qubit 2
    # - qubit 0: final split, conditioned on qubits 2 and 1
    circuit.ry(angles[0], 2)

    # Anti-controlled rotation: applied when control qubit 2 is in state |0>
    if 1 > 0:
        circuit.cry(angles[1], 2, 1, ctrl_state="0")

    # Controlled rotation: applied when control qubit 2 is in state |1>
    if 1 > 0:
        circuit.cry(angles[2], 2, 1)

    # Two-control rotations on the last target qubit (qubit 0)
    if 1 > 0:
        c2ry = RYGate(angles[3]).control(2, ctrl_state="00")
        circuit.append(c2ry, [2, 1, 0])

        c2ry = RYGate(angles[4]).control(2, ctrl_state="10")
        circuit.append(c2ry, [2, 1, 0])

        c2ry = RYGate(angles[5]).control(2, ctrl_state="01")
        circuit.append(c2ry, [2, 1, 0])

        c2ry = RYGate(angles[6]).control(2)
        circuit.append(c2ry, [2, 1, 0])

    return circuit

print("Function defined: gen_circuit(angles)")
print("This function assembles the corresponding 3-qubit Qiskit circuit.")

## Step 2, choose the target amplitudes and build the circuit

We now specify the real-valued amplitudes to be loaded into the 3-qubit register.

Because a 3-qubit state has $2^3 = 8$ computational-basis states, the target vector must contain **8 entries**.  
The vector below is already normalized and is interpreted directly as an **amplitude vector**:

$x = [x_0, x_1, \ldots, x_7]$.

The next cell:

- prints the target amplitudes,
- computes the recursive angles,
- builds the corresponding circuit,
- and displays the circuit diagram.


In [ ]:
# Target real-valued amplitudes to be loaded in the computational basis
p = [
    0.169194173824,
    0.187500000000,
    0.169194173824,
    0.125000000000,
    0.080805826176,
    0.062500000000,
    0.080805826176,
    0.125000000000,
]

print("Target amplitude vector x (8 entries for 3 qubits):")
print(np.array(p))

# In this notebook p already stores amplitudes, not probabilities
x = np.array(p, dtype=float)

# Compute the recursive rotation angles
angles = gen_angles(x)
print("\nRotation angles returned by gen_angles(x):")
for i, angle in enumerate(angles):
    print(f"  angle[{i}] = {angle:.6f} rad")

# Build and display the state-preparation circuit
circuit = gen_circuit(angles)
print("\n3-qubit state-preparation circuit:")
circuit.draw("mpl")

## Step 3, inspect the ideal state prepared by the circuit

Before adding measurements, it is useful to check the **exact statevector** produced by the circuit.

This allows us to verify, in an ideal noiseless setting, that the circuit prepares the expected amplitudes.  
The next cell prints, for each computational-basis state:

- the basis label,
- the complex amplitude,
- and the corresponding probability.


In [ ]:
from qiskit.quantum_info import Statevector

# Ideal statevector corresponding to the preparation circuit
statevector = Statevector.from_instruction(circuit)
ideal_distribution = statevector.probabilities_dict()

print("Ideal probabilities from the exact statevector:")
print(ideal_distribution)

# Print amplitudes and probabilities in computational-basis order
amps = statevector.data
num_qubits = statevector.num_qubits

print("\nStatevector entries in computational-basis order:")
for i, amp in enumerate(amps):
    bitstring = format(i, f"0{num_qubits}b")
    print(f"|{bitstring}>: amplitude = {amp:.6f}, probability = {abs(amp)**2:.6f}")

## Step 4, add measurements and transpile the circuit

To compare the prepared state with a sampled output distribution, we now add measurements on all qubits.

The transpilation step converts the circuit into a form compatible with the selected backend.  
Here we use Qiskit's `BasicSimulator`, which is sufficient for this pedagogical example.


In [ ]:
# Add measurements on all qubits
circuit.measure_all()
print("Measurements added on all qubits.")

# Transpile to a basis of one- and two-qubit gates for the selected backend
backend = BasicSimulator()
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
circuit_combine = pm.run(circuit)

print("\nTranspiled circuit executed on BasicSimulator:")
circuit_combine.draw("mpl")

## Step 5, compare sampled counts with the target distribution

Finally, we execute the circuit with a finite number of shots and compare the resulting histogram with the expected target distribution.

A small but important note concerns **bit ordering**:

- many physics texts write basis states in **big-endian** order, such as $|q_0 q_1 q_2\rangle$,
- while Qiskit internally uses **little-endian** conventions in several contexts.

For this reason, it is good practice to state explicitly which ordering is being used whenever basis labels are compared.


In [ ]:
# Reminder: p currently stores amplitudes
print("Target amplitudes x:")
print(np.array(p))

# Execute the transpiled circuit
N_shots = 100000
print(f"\nNumber of shots used for sampling: {N_shots}")

# Note on bit ordering:
# many physics references use the big-endian convention |q0 q1 ... q(n-1)>,
# while Qiskit often uses little-endian ordering internally.
# If needed, the next line can be used to reverse the displayed bit order:
# result = backend.run(circuit_combine.reverse_bits(), shots=N_shots).result()

result = backend.run(circuit_combine, shots=N_shots).result()
counts = result.get_counts()

# Convert amplitudes into target probabilities
amplitudes = np.array(p, dtype=float)
p = amplitudes**2
valth = N_shots * p

print("\nTarget probabilities x_i^2:")
for i, prob in enumerate(p):
    print(f"  state {i:03b}: {prob:.6f}")

countsth = {
    "000": int(valth[0]),
    "001": int(valth[1]),
    "010": int(valth[2]),
    "011": int(valth[3]),
    "100": int(valth[4]),
    "101": int(valth[5]),
    "110": int(valth[6]),
    "111": int(valth[7]),
}

pth = {
    "000": p[0],
    "001": p[1],
    "010": p[2],
    "011": p[3],
    "100": p[4],
    "101": p[5],
    "110": p[6],
    "111": p[7],
}

print("\nMeasured counts from the simulator:")
print(counts)

legend = ["Target", "Uploaded"]

# Uncomment one of the following plots as needed:
# plot_histogram(counts, title="3-qubit real data loading in Qiskit")
# plot_histogram([countsth, counts], legend=["Target counts", "Measured counts"],
#                title="3-qubit real data loading in Qiskit")
plot_histogram(
    [pth, counts],
    legend=legend,
    title="3-qubit real data loading in Qiskit"
)
